# Extending the analysis to 3 dimensions
---

*Introduction to Image Analysis Workshop*

*Stefania Marcotti (stefania.marcotti@crick.ac.uk)*

*Counting objects in 3 dimensions*

*CC-BY-SA-4.0 license: creativecommons.org/licenses/by-sa/4.0/*

*Adapted from Tom Slater (slatert2@cardiff.ac.uk)*

*[Intro to building image analysis pipelines for 3D data with Python](https://github.com/RMS-DAIM/introduction-to-image-analysis/blob/main/Scripts/Jupyter/3d_analysis.ipynb)*

---

## Introduction

In many microscopy experiments, images are not collected as single 2D images, but instead as stacks of images representing a 3D volume.

Examples include:

- Confocal microscopy
- X-ray tomography
- Electron tomography
- FIB-SEM serial sectioning

In this notebook, we will explore how standard 2D segmentation concepts can be extended into 3D using `scikit-image`.

We will cover:

- Understanding 3D image data
- Visualising slices through a volume
- Thresholding 3D data
- 3D connected component labelling
- Measuring segmented 3D objects

This notebook is inspired by the official [`scikit-image` tutorial on 3D image processing](https://scikit-image.org/skimage-tutorials/lectures/three_dimensional_image_processing.html).

## Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy import ndimage as ndi

from skimage import data
from skimage import filters
from skimage import measure
from skimage import morphology
from skimage import segmentation
from skimage import feature
from skimage import io

import pandas as pd

import wget

import napari

## Download data

One thing we can do with Python is download data directly from databases. Here, we use the [`wget` library](https://pypi.org/project/wget/) to download directly from an online repository, [Zenodo](https://zenodo.org/). Zenodo is a general-purpose open repository which allows researchers to deposit research papers, datasets, research software, reports, and any other research-related digital artefacts. For each submission, a persistent digital object identifier (DOI) is minted, which makes the stored items easily citeable.

In [ ]:
wget.download('https://zenodo.org/records/20268094/files/AuPdGr_T3_HAADF_1_SIRT20i_1n.tif','../../Data/AuPdGr_T3_HAADF_1_SIRT20i_1n.tif')

## Open images
We can open images in the same way as our 2D images, using the `imread` function from `scikit-image.io`. Here, we open an electron tomography dataset of Au and Pd nanoparticles.

In [ ]:
vol = io.imread("../../Data/AuPdGr_T3_HAADF_1_SIRT20i_1n.tif")

In [ ]:
print('Image dimensions:', vol.shape)

`skimage` doesn't read metadata! We can open `napari` to visualise the data and figure out the order of the dimensions.

In [ ]:
viewer = napari.Viewer()
viewer.add_image(vol, blending='additive')

We can see that the data is three-dimensional: it has 440 slices, each with a dimension of 436 x 504 pixels. We can therefore conclude that the data was opened as (Z,Y,X) as per convention.

We could also plot using `matplotlib.pyplot` some slices in Z.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

slices = [50, 100, 150, 200, 250, 300]

for ax, z in zip(axes.ravel(), slices):
    ax.imshow(vol[z], cmap='gray')
    ax.set_title(f'z = {z}')
    ax.axis('off')

## Use filters to suppress noise
If you have noisy data, filters can be applied in much the same manner as for 2D images. The filters in `scikit-image` work on arrays of arbitrary dimensions, so the Gaussian filter shown below uses a three-dimensional Gaussian kernel. You could apply a 2D Gaussian to each slice instead, but you wouldn't be taking into account the intensities in neighbouring Z-slices.

In [ ]:
vol_gauss = filters.gaussian(vol, sigma=2)

# display original image next to filtered one
fig, axs = plt.subplots(1, 2, figsize=(6,4))

axs[0].imshow(vol[50], cmap='gray')
axs[0].set_title('Slice of original volume')
axs[0].axis('off')

axs[1].imshow(vol_gauss[50], cmap='gray')
axs[1].set_title('Filtered slice (Gaussian filter)')
axs[1].axis('off')

plt.tight_layout()

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
It's also possible to use a different value for sigma in each direction, which may be useful if you have anisotropic data. Using the templates below, create a new volume called <code>vol_gauss_test</code> with a different value for sigma for each dimension (<code>sigma=(Z,Y,X)</code>) and display a slice next to a slice from the original volume - what do you notice? </div>

In [ ]:
# Gaussian blur (sigma=(?,?,?) YOU DECIDE!)
vol_gauss_test = [...]

In [ ]:
# display slice from original volume next to filtered one (vol_gauss_test)
fig, axs = plt.subplots(1, 2, figsize=(6,4))

axs[0].imshow(vol[50], cmap='gray')
axs[0].set_title('Slice from original volume')
axs[0].axis('off')

# add your code to visualise the new image here

plt.tight_layout()

## Segment volumes
When it comes to segmentation, this is again directly translated from 2D to 3D so our code looks remarkably similar. For example, to use an Otsu threshold we use the same <code>threshold_otsu</code> function we used in the 2D example.

In [ ]:
thresh = filters.threshold_otsu(vol_gauss)

vol_thresh = vol_gauss > thresh

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
Add the segmented volume to the <code>napari</code> viewer for visualisation </div>

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
Try another of the thresholding methods that you explored in 2D  </div>

In [ ]:
# Choose a different thresholding method among the ones displayed above!
thresh_other = [...]
vol_thresh_other = [...]

# add to napari viewer

## Counting objects
Again, we can directly translate our code for labelling objects from 2D to 3D, i.e. we use the same <code>skimage.measure.label</code> function.

In [ ]:
# label objects and visualise the result
labels = measure.label(vol_thresh)

# count the objects - find the maximum integer assigned to a label!
print('There are', labels.max(), 'objects in the image')

In [ ]:
viewer.add_labels(labels, blending='additive')

## Morphological quantification
Morphological quantification is similar in 3D to 2D, although our considerations are slightly different. For a start, not all of the properties are calculable in 3D, so we're not able to include properties such as eccentricity. Let's remove eccentricity from our request to `regionprops`, but otherwise we can use the same code as in 2D.

In [ ]:
# measure properties
props = measure.regionprops_table(labels, properties=['label', 'area', 'centroid'])
props_df = pd.DataFrame(props)

props_df.head()

Note that we now have 3 centroid values corresponding to our 3 dimensions (rather than 2). It's also important to note that area is a misnomer, the value for area actually represents the number of voxels in each region and is therefore our volume given in units of voxels.

In [ ]:
# Plot a histogram of volumes
regions = measure.regionprops(labels)
volumes = [region.area for region in regions]

plt.hist(volumes, bins=40)
plt.xlabel('Volume (voxels)')
plt.ylabel('Count')
plt.title('Distribution of object volumes')

plt.tight_layout()

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
You know that eccentricity is not available as a property of 3D labels, but which others are? Construct a new regionprops table in the cell below and try some of the different properties listed <a href="https://scikit-image.org/docs/0.25.x/api/skimage.measure.html#skimage.measure.regionprops">here</a>. </div>

In [ ]:
new_props = [...]
new_props_df = pd.DataFrame(new_props)

new_props_df.head()